In [1]:
import pandas as pd
from elasticsearch import Elasticsearch, helpers
import re
import json
import requests

In [3]:
from snowflake.snowpark.session import Session
 
connection_params = {
    "user": "mohankrishna.samavedam@celanese.com",
    "authenticator": "externalbrowser",
    "account": "celanese-celanytics.privatelink",
    "warehouse": "reporting_wh",
    "database": "analytics_dev",
    # "database": "analytics_qa", #use for skipping the new brands updated in the Dev #F-hot-fix
    "schema": "snowpark",
    "role": "data_developer_gst"  }
snowpark_session = Session.builder.configs(connection_params).create()

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/7a3c88ff-a5f6-449d-ac6d-e8e3aa508e37/saml2?SAMLRequest=pZPNctowFIVfxaOubcsGE0eDydBQGjppy4DTn%2BxU%2BRoUZMmRZAx5%2BsoGZtJFsunKGulcfefeI49vDpXw9qANVzJDUYCRB5KpgstNhh7yuZ8iz1gqCyqUhAwdwaCbydjQStRk2titXMFzA8Z67iJpSHeQoUZLoqjhhkhagSGWkfX06z2JA0yoMaCtw6FzSWG4Y22trUkYtm0btINA6U0YY4xDfB06VSf5gF4h6vcZtVZWMSUuJQfX0xuIKMTDDuEUjrA8F37k8jSC9yh%2FTiJD7vJ86S%2B%2Fr3PkTS%2Fd3Sppmgr0GvSeM3hY3Z8MGOeAgaASDPj94mg5M0Gt%2BZ5aEFzuAiNVWwq6A6aqurGOEbhVWEIRCrXhbnKLWYbqHS%2BiaP3UbofzdhWpx0T%2B2kSfDvndZ8yrZfTl98vTzxebbA91o55Thrwfl5zjLueFMQ0sZJeudVs4Hvk49fEgxyMSj0iCg0ESPSJv5tLlktq%2B8tJC7yOoONPKqNIq6YxD7%2FKKDlialqVPk3LkD4fXhU%2FZqPAhhQGlCXafq7DLMEand0R6I3ryf9MZh6%2FvOj%2FQby6zxWypBGdHb650Re3bkUZB1O%2Fwwi97KYGKcjEtCg3GuGiFUO2tBucjQ1Y3gMLJifrvnzD5Cw%3D%3D&RelayState=64120 to authenticate...
A browser window should have opened for you to complete the login. If you can't see it

 pip install snowflake-connector-python[secure-local-storage]


In [4]:
color_df = snowpark_session.table('GST_CURATED.AUSP_SAP_MATERIAL_COLOR_CODE').to_pandas()
color_df_sub = color_df[['COLOR_ID','COLOR_DESC']].dropna().reset_index(drop=True)

In [ ]:
color_codes = list(set(color_df['COLOR_NAME'].unique().tolist()+color_df['COLOR_DESC'].unique().tolist()+color_df['COLOR_ID'].unique().tolist() + (color_df_sub['COLOR_ID'] +" " +color_df_sub['COLOR_DESC']).tolist()))
color_codes = [i.lower() for i in color_codes if i is not None]
color_codes = [i for i in color_codes if len(i)>1]
color_codes = [i for i in color_codes if  i not in ['n/a']]
color_codes = [i for i in color_codes if 'for' not in i]
color_codes.extend(['bk010','bk10','dk brown'])
color_codes = list(set(color_codes))
color_codes.remove('pa6.6')

In [ ]:
# import re  
  
# def generate_pattern_from_code(code):  
#     # Split the code into parts and create a flexible pattern to match any specified separator.  
#     parts = re.split(r'[-./\s()]', code)  
#     pattern_parts = [re.escape(part) for part in parts]  
#     # Join parts with a regex pattern that matches any of the specified separators.  
#     return '[-./\s()]*'.join(pattern_parts)  
  
# # color_codes = ['80/5763', 'H119 SMOKE', '7245 (7043)', 'NT205375','000']  
# patterns = []  
  
# for code in color_codes:  
#     patterns.append(generate_pattern_from_code(code))  
  
# # Add directly the parts of color codes if they're significant (length > 2)  
# for code in color_codes:  
#     parts = re.split(r'[-./\s()]', code)  
#     for part in parts:  
#         if len(part) > 2:  
#             patterns.append(re.escape(part))  
  
# # Remove duplicates  
# patterns = list(set(patterns))  
  
# # Join all patterns into a single regex pattern, preparing for variations at the end of strings  
# pattern = '[\s\(-](' + '|'.join(patterns) + ')[-./\s()]*$'  

In [7]:
import re  
  
def generate_pattern_from_code(code):  
    # Split the code into parts and create a flexible pattern to match any specified separator.  
    parts = re.split(r'[-./\s()]', code)  
    pattern_parts = [re.escape(part) for part in parts]  
    # Join parts with a regex pattern that matches any of the specified separators.  
    return '[-./\s()]*'.join(pattern_parts)  
  
# color_codes = ['80/5763', 'H119 SMOKE', '7245 (7043)', 'NT205375', '000']  
patterns = []  
  
for code in color_codes:  
    generated_pattern = generate_pattern_from_code(code)  
    # Pattern to match the code at the end or as the entire string  
    patterns.append(f"(?:{generated_pattern})")  
    # Also add a pattern to match the code exactly as the entire string  
    patterns.append(f"^{generated_pattern}$")  
  
# Add directly the parts of color codes if they're significant (length > 2)  
for code in color_codes:  
    parts = re.split(r'[-./\s()]', code)  
    for part in parts:  
        if len(part) > 2:  
            escaped_part = re.escape(part)  
            patterns.append(escaped_part)  
            # Also add a pattern to match the part exactly as the entire string  
            patterns.append(f"^{escaped_part}$")  
  
# Remove duplicates  
patterns = list(set(patterns))  
  
# Join all patterns into a single regex pattern, preparing for variations at the end of strings or matching the entire string  
pattern = '([\s\(-]|^)(' + '|'.join(patterns) + ')[-./\s()]*$'  
  
# The final pattern will match codes at the end of a string or exactly as the entire string.  


In [ ]:
with open("../dependencies/oos_color_code_pattern.txt", 'w') as file:  
    file.write(pattern)

In [ ]:
with open("../dependencies/oos_color_code_pattern.txt", 'r') as file:  
    pattern = file.read()

In [ ]:
# [i for i in patterns if "800" in i]

In [11]:
# [i for i in color_codes if "800" in i]

In [12]:
%%time
strings = [  
    'Product Color 80/5763',  
    'Another Product 5763',  
    'Example H119 SMOKE',  
    'Test Product NT205375',  
    'With Variant 7245-7043',  
    'Another Variant 7043',  
    'Yet Another Variant 7245/7043',  
    'Yet Another Variant 8000',  
    'No Match Here'  ,
    "This is a test NT205375", "NT205375", "Some text (7245/7043)", "7245", "H119-SMOKE",
    "zytel pa6"
]  
  
for string in strings:  
    # Note: Using re.IGNORECASE to ensure case-insensitive matching  
    new_s = re.sub(pattern, '', string, flags=re.IGNORECASE).rstrip()  
    print(f'Original: "{string}" -> Processed: "{new_s}"') 

Original: "Product Color 80/5763" -> Processed: "Product Color"
Original: "Another Product 5763" -> Processed: "Another Product"
Original: "Example H119 SMOKE" -> Processed: "Example"
Original: "Test Product NT205375" -> Processed: "Test Product"
Original: "With Variant 7245-7043" -> Processed: "With Variant"
Original: "Another Variant 7043" -> Processed: "Another Variant"
Original: "Yet Another Variant 7245/7043" -> Processed: "Yet Another Variant"
Original: "Yet Another Variant 8000" -> Processed: "Yet Another Variant"
Original: "No Match Here" -> Processed: "No Match Here"
Original: "This is a test NT205375" -> Processed: "This is a test"
Original: "NT205375" -> Processed: ""
Original: "Some text (7245/7043)" -> Processed: "Some text"
Original: "7245" -> Processed: ""
Original: "H119-SMOKE" -> Processed: ""
Original: "zytel pa6" -> Processed: "zytel pa6"
CPU times: total: 3.56 s
Wall time: 3.8 s


In [13]:
%%time
string= "lx2002"
new_s = re.sub(pattern, '', string.lower()).rstrip()
print(f'Original: "{string}" -> Processed: "{new_s}"')

Original: "lx2002" -> Processed: ""
CPU times: total: 3.12 s
Wall time: 3.53 s


In [14]:
[i for i in patterns if "lx2002" in i]

['lx2002',
 '^lx2002$',
 '(?:gray[-./\\s()]*lx2002)',
 '^gray[-./\\s()]*lx2002$',
 '(?:lx2002)']

In [ ]:
[i for i in color_codes if "lx2002" in i]

['lx2002', 'gray lx2002']

In [ ]:
value_to_check = "lx2002"
if any(value_to_check in substring.lower() for substring in color_df['COLOR_NAME'].unique().tolist() if substring is not None):
    print("identified in COLOR_NAME")
if any(value_to_check in substring.lower() for substring in color_df['COLOR_DESC'].unique().tolist() if substring is not None):
    print("identified in COLOR_DESC")
if any(value_to_check in substring.lower() for substring in color_df['COLOR_ID'].unique().tolist() if substring is not None):
    print("identified in COLOR_ID")
                    

identified in COLOR_DESC


In [17]:
for i in color_df['COLOR_DESC']:
    if i!=None and value_to_check in i.lower():
        print(i)
        

LX2002
